# BRECHA CATASTRAL Y ASENTAMIENTOS INFORMALES · HONDURAS
## Kit Replicable para Gobiernos Locales y Gestión Territorial
### Clústers Copernicus Centroamérica – Fase II · Ficha N.° 8

**Zona Piloto:** Mancha urbana consolidada del Municipio del Distrito Central (Tegucigalpa y Comayagüela)

**Equipo Técnico:**
- **Mario Baide** · Universidad Nacional Autónoma de Honduras (UNAH) — *Profesor Investigador*
- **Mario Matamoros** · Universidad Nacional Autónoma de Honduras (UNAH) — *Profesor Investigador*
- **Jonathan Ramirez** · Alcaldía Municipal del Distrito Central (AMDC) — *Analista en Planificación Urbana*
- **Ana Maritza Irías** · Secretaría de Gobernación, Justicia y Descentralización (SEGOB) — *Especialista institucional*

---
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mariobaidemunoz-del/urbano-hn-distrito-central/blob/main/notebooks/brecha_catastral_distrito_central.ipynb)

## 1. Instalación de librerías y dependencias
Instalamos `geemap`, `earthengine-api`, `geopandas` y herramientas de análisis espacial para Google Colab.

In [ ]:
!pip install -q geemap earthengine-api geopandas matplotlib

## 2. Autenticación e Inicialización de Google Earth Engine
Ejecuta la siguiente celda e introduce tus credenciales de Google cuando se solicite.

In [ ]:
import ee
import geemap
import math
import json
import geopandas as gpd
import matplotlib.pyplot as plt

# Inicialización con manejo de autenticación en Colab
try:
    ee.Initialize()
    print("✔ Google Earth Engine inicializado correctamente.")
except Exception as e:
    ee.Authenticate()
    ee.Initialize()
    print("✔ Google Earth Engine autenticado e inicializado con éxito.")

## 3. Parámetros de Configuración (`CONFIG`)
Permite parametrizar el análisis para el **Distrito Central** o replicarlo en cualquier municipio de Honduras.

In [ ]:
CONFIG = {
    'pais': 'HND',
    'municipio': 'Distrito Central',      # Nombre en geoBoundaries ADM2
    'ambito': 'mancha',                    # 'mancha' = consolidada · 'municipio' = todo el municipio
    'umbralManchaM2': 1000,                # m² construidos por celda GHSL de 1 ha para considerarla urbana
    'areaMinManchaHa': 50,                 # Descarta núcleos aislados menores a 50 ha
    'anioInicio': 2016,
    'anioFin': 2026,
    'mesesSecos': [1, 4],                  # Enero a Abril (estación seca)
    'confianzaEdif': 0.70,                 # Umbral de confianza Open Buildings v3
    'umbralPresenciaOB': 0.50,             # Umbral building_presence (2.5D Temporal)
    'pendienteCritica': 30,                # % de pendiente considerada crítica
    'tamCelda': 250,                       # Retícula de análisis (metros)
    'epsg': 'EPSG:32616',                  # UTM 16N (Honduras)
    'densidadInformal': 40,                # Edificaciones por hectárea
    'areaMediaInformal': 80,               # m² medios por edificación
    'umbralLadera': 0.30,                  # Proporción en pendiente crítica o riesgo
    'umbralNuevo': 0.50,                   # Proporción edificación aparecida tras año base
    'umbralFueraCatastro': 0.50
}
print(f"Configuración cargada para {CONFIG['municipio']}, {CONFIG['pais']}.")

## 4. Delimitación Territorial
Obtenemos el polígono municipal oficial mediante geoBoundaries y delimitamos la **mancha urbana consolidada** a partir de la huella histórica de GHSL (cierre morfológico).

In [ ]:
def obtener_municipio(nombre, pais='HND'):
    adm2 = ee.FeatureCollection('WM/geoLab/geoBoundaries/600/ADM2').filter(ee.Filter.eq('shapeGroup', pais))
    return adm2.filter(ee.Filter.eq('shapeName', nombre)).geometry()

def obtener_ambito(nombre, config):
    muni = obtener_municipio(nombre, config['pais'])
    if config.get('ambito') == 'municipio':
        return muni
    ghsl = ee.Image(ee.ImageCollection('JRC/GHSL/P2023A/GHS_BUILT_S')
                    .filterDate('2020-01-01', '2021-01-01').first()).select('built_surface').clip(muni)
    urbano = ghsl.gte(config['umbralManchaM2'])\
                 .focalMax(1, 'square', 'pixels')\
                 .focalMin(1, 'square', 'pixels')\
                 .selfMask()
    poligonos = urbano.reduceToVectors(
        geometry=muni, scale=100, crs=config['epsg'], geometryType='polygon',
        eightConnected=True, labelProperty='urbano', maxPixels=1e10
    )
    filtrados = poligonos.map(lambda f: f.set('ha', f.geometry().area(10).divide(10000)))\
                         .filter(ee.Filter.gte('ha', config['areaMinManchaHa']))
    return filtrados.geometry(10)

aoi = obtener_ambito(CONFIG['municipio'], CONFIG)
print("✔ Ámbito espacial (mancha urbana consolidada) obtenido.")

## 5. Modelado de la "Ciudad Construida"
- **Sentinel-2 L2A**: Compuesto de mediana de estación seca con máscara Cloud Score+.
- **Índices espectrales**: NDBI (construido), NDVI (vegetación) y BSI (suelo desnudo).
- **Dynamic World**: Detección de consolidación urbana (2016–2026).
- **GHSL**: Evolución histórica a largo plazo (1975–2020).
- **Open Buildings v3 y 2.5D Temporal**: Huellas individuales y año de aparición (2016–2023).

In [ ]:
def composite_s2(aoi, anio, config):
    cs = ee.ImageCollection('GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED')
    inicio = ee.Date.fromYMD(anio, config['mesesSecos'][0], 1)
    fin = ee.Date.fromYMD(anio, config['mesesSecos'][1], 1).advance(1, 'month')
    
    def mascara_nubes(img):
        return img.updateMask(img.select('cs_cdf').gte(0.60))\
                  .select(['B2', 'B3', 'B4', 'B8', 'B11', 'B12'])\
                  .divide(10000)
                  
    return ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')\
             .filterBounds(aoi).filterDate(inicio, fin)\
             .linkCollection(cs, ['cs_cdf'])\
             .map(mascara_nubes).median().clip(aoi)

def indices_espectrales(img):
    ndvi = img.normalizedDifference(['B8', 'B4']).rename('NDVI')
    ndbi = img.normalizedDifference(['B11', 'B8']).rename('NDBI')
    bsi = img.expression(
        '((SWIR + RED) - (NIR + BLUE)) / ((SWIR + RED) + (NIR + BLUE))',
        {'SWIR': img.select('B11'), 'RED': img.select('B4'), 'NIR': img.select('B8'), 'BLUE': img.select('B2')}
    ).rename('BSI')
    return ndvi.addBands(ndbi).addBands(bsi)

def construido_anual(aoi, anio, config):
    inicio = ee.Date.fromYMD(anio, config['mesesSecos'][0], 1)
    fin = ee.Date.fromYMD(anio, config['mesesSecos'][1], 1).advance(1, 'month')
    return ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1')\
             .filterBounds(aoi).filterDate(inicio, fin)\
             .select('label').mode().eq(6)\
             .rename('construido').set('anio', anio).clip(aoi)

def anio_consolidacion(aoi, config):
    anios = ee.List.sequence(config['anioInicio'], config['anioFin'] - 1)
    def consolidar_ano(a):
        a = ee.Number(a)
        hoy = construido_anual(aoi, a, config)
        sig = construido_anual(aoi, a.add(1), config)
        estable = hoy.And(sig)
        return ee.Image.constant(a).toInt16().updateMask(estable).rename('anio')
    imgs = anios.map(consolidar_ano)
    return ee.ImageCollection.fromImages(imgs).min().clip(aoi)

def historia_ghsl(aoi):
    ghsl = ee.ImageCollection('JRC/GHSL/P2023A/GHS_BUILT_S')
    epocas = [1975, 1980, 1985, 1990, 1995, 2000, 2005, 2010, 2015, 2020]
    def mapear_epoca(e):
        e = ee.Number(e)
        d1 = ee.Date.fromYMD(e, 1, 1)
        d2 = ee.Date.fromYMD(e.add(1), 1, 1)
        img = ee.Image(ghsl.filterDate(d1, d2).first()).select('built_surface')
        return ee.Image.constant(e).toInt16().updateMask(img.gte(1000)).rename('epoca')
    imgs = ee.List(epocas).map(mapear_epoca)
    return ee.ImageCollection.fromImages(imgs).min().clip(aoi)

def edificaciones(aoi, config):
    return ee.FeatureCollection('GOOGLE/Research/open-buildings/v3/polygons')\
             .filterBounds(aoi).filter(ee.Filter.gte('confidence', config['confianzaEdif']))

def anio_aparicion_edif(aoi, config):
    ob = ee.ImageCollection('GOOGLE/Research/open-buildings-temporal/v1').filterBounds(aoi)
    anios = ee.List.sequence(2016, 2023)
    def mapear_ob(a):
        a = ee.Number(a)
        anual = ob.filter(ee.Filter.calendarRange(a, a, 'year')).mosaic()
        presente = anual.select('building_presence').gte(config['umbralPresenciaOB'])
        return ee.Image.constant(a).toInt16().updateMask(presente).rename('anio_edif')
    imgs = anios.map(mapear_ob)
    return ee.ImageCollection.fromImages(imgs).min().clip(aoi)

print("✔ Funciones de ciudad construida definidas.")

## 6. Condicionantes Críticos y Pendiente (Copernicus DEM GLO-30)
Cálculo de pendientes en porcentaje a partir del Modelo Digital de Elevación de Copernicus a 30 metros de resolución.

In [ ]:
def condicionantes(aoi, config):
    dem_col = ee.ImageCollection('COPERNICUS/DEM/GLO30').filterBounds(aoi).select('DEM')
    dem = dem_col.mosaic().setDefaultProjection(dem_col.first().projection())
    pendiente_grados = ee.Terrain.slope(dem)
    pendiente_pct = pendiente_grados.multiply(math.pi / 180).tan().multiply(100)
    critica = pendiente_pct.gte(config['pendienteCritica'])
    return critica.rename('condicion_critica').clip(aoi)

print("✔ Condicionantes de pendiente crítica definidos.")

## 7. Retícula de Análisis, Cruce y Tipología A–E
Se genera la retícula de 250 m (o colonias si se dispone de la capa municipal), se calculan las métricas morfológicas (densidad, tamaño medio) y se asigna la tipología correspondiente.

In [ ]:
def unidades_de_analisis(aoi, config):
    proj = ee.Projection(config['epsg']).atScale(config['tamCelda'])
    return aoi.coveringGrid(proj).map(lambda f: f.set('unidad', f.id()))

def calcular_indicadores(aoi, config):
    edif = edificaciones(aoi, config)
    huella_edif = ee.Image(0).byte().paint(edif, 1).rename('edif').clip(aoi)
    aparicion = anio_aparicion_edif(aoi, config)
    nuevo = aparicion.gte(config['anioInicio'] + 1).unmask(0).And(huella_edif).rename('edif_nueva')
    critica = condicionantes(aoi, config)
    
    conteo = edif.map(lambda f: ee.Feature(f.centroid(1)).set('uno', 1))\
                 .reduceToImage(['uno'], ee.Reducer.count()).unmask(0).rename('n_edif')
                 
    area = ee.Image.pixelArea()
    pila = huella_edif.multiply(area).rename('m2_edif')\
                      .addBands(nuevo.multiply(area).rename('m2_nuevo'))\
                      .addBands(huella_edif.And(critica).multiply(area).rename('m2_critico'))\
                      .addBands(conteo)
                      
    unidades = unidades_de_analisis(aoi, config)
    stats = pila.reduceRegions(
        collection=unidades, reducer=ee.Reducer.sum(), scale=4, crs=config['epsg'], tileScale=8
    )
    
    def tipificar(f):
        ha = f.geometry().area(1).divide(10000)
        m2 = ee.Number(f.get('m2_edif'))
        hay = m2.gt(0)
        den = m2.max(1)
        n = ee.Number(f.get('n_edif'))
        p_nuevo = ee.Number(f.get('m2_nuevo')).divide(den)
        p_critico = ee.Number(f.get('m2_critico')).divide(den)
        densidad = n.divide(ha)
        area_media = m2.divide(n.max(1))
        morf_informal = densidad.gte(config['densidadInformal']).And(area_media.lte(config['areaMediaInformal']))
        
        tipo = ee.Algorithms.If(hay.Not(), 'Sin edificación',
               ee.Algorithms.If(morf_informal.And(p_critico.gte(config['umbralLadera'])), 'Candidato informal en condición crítica',
               ee.Algorithms.If(morf_informal.And(p_nuevo.gte(config['umbralNuevo'])), 'Candidato informal reciente',
               ee.Algorithms.If(morf_informal, 'Candidato informal consolidado',
               'Tejido no clasificado como informal'))))
        
        props = {
            'ha': ha,
            'n_edif': n,
            'dens_edif_ha': densidad,
            'area_media_m2': area_media,
            'cobertura_edif': m2.divide(ha.multiply(10000)),
            'p_nuevo': p_nuevo,
            'p_critico': p_critico,
            'morf_informal': morf_informal,
            'tipologia': tipo
        }
        return f.set(props)
        
    tabla = stats.map(tipificar)
    return {
        'edif': edif, 'huella_edif': huella_edif, 'aparicion': aparicion,
        'nuevo': nuevo, 'critica': critica, 'tabla': tabla
    }

print("✔ Módulo de indicadores zonal y tipología configurado.")

## 8. Ejecución del Procesamiento Espacial
Generamos las capas analíticas para la zona del Distrito Central.

In [ ]:
print("Procesando capas multitemporales para el Distrito Central...")
s2_composite = composite_s2(aoi, CONFIG['anioFin'], CONFIG)
idx = indices_espectrales(s2_composite)
consol = anio_consolidacion(aoi, CONFIG)
ghsl = historia_ghsl(aoi)
resultado = calcular_indicadores(aoi, CONFIG)
print("✔ Capas y tabla de indicadores generadas exitosamente.")

## 9. Visualizador Interactivo con `geemap`
Explora todas las capas satelitales, índices, huellas de edificaciones y la tipología de asentamientos informales directamente en el mapa.

In [ ]:
Map = geemap.Map()
Map.centerObject(aoi, 12)

pal_anios = ['#fff5eb', '#fdd0a2', '#fd8d3c', '#d94801', '#7f2704']

# 1. Imagen Sentinel-2 RGB
Map.addLayer(s2_composite, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 0.25}, 'Sentinel-2 RGB 2026')

# 2. Índice NDBI (Índice de Edificación Normalizado)
Map.addLayer(idx.select('NDBI'), {'min': -0.3, 'max': 0.3, 'palette': ['#1a9850', '#ffffbf', '#d73027']}, 'NDBI (Construido)', False)

# 3. GHSL Épocas (1975–2020)
Map.addLayer(ghsl, {'min': 1975, 'max': 2020, 'palette': pal_anios}, 'GHSL Huella Histórica (1975–2020)', False)

# 4. Consolidación Dynamic World
Map.addLayer(consol, {'min': CONFIG['anioInicio'], 'max': CONFIG['anioFin'] - 1, 'palette': pal_anios}, 'Año de Consolidación (Dynamic World)', False)

# 5. Aparición Open Buildings
Map.addLayer(resultado['aparicion'], {'min': 2016, 'max': 2023, 'palette': pal_anios}, 'Año Aparición Edificaciones (OB 2.5D)', False)

# 6. Huellas individuales de edificaciones
Map.addLayer(resultado['edif'].style(color='444444', width=0.5, fillColor='00000000'), {}, 'Edificaciones (Open Buildings v3)', False)

# 7. Condición crítica (Pendiente >= 30%)
Map.addLayer(resultado['critica'].selfMask(), {'palette': ['#6a51a3'], 'opacity': 0.45}, 'Pendiente Crítica (>= 30%)', False)

# 8. Ámbito territorial (Mancha Urbana)
Map.addLayer(ee.Image().byte().paint(ee.FeatureCollection([ee.Feature(aoi)]), 1, 2), {'palette': ['#FFC800']}, 'Mancha Urbana Consolidada')

# 9. Tipología de candidatos a asentamiento informal
tipos = [
    'Tejido no clasificado como informal',
    'Candidato informal consolidado',
    'Candidato informal reciente',
    'Candidato informal en condición crítica'
]
colores = ['#d9d9d9', '#f46d43', '#d73027', '#67001f']

codigo = resultado['tabla'].filter(ee.Filter.neq('tipologia', 'Sin edificación'))\
                           .map(lambda f: f.set('cod', ee.List(tipos).indexOf(f.get('tipologia'))))
img_tipo = ee.Image().byte().paint(codigo, 'cod')
Map.addLayer(img_tipo, {'min': 0, 'max': len(tipos) - 1, 'palette': colores}, 'Tipología de Asentamientos')

# Leyenda explicativa
legend_dict = dict(zip(tipos, colores))
Map.add_legend(title="Tipología Urbana", legend_dict=legend_dict)

Map

## 10. Análisis Estadístico de la Brecha y Asentamientos
Extraemos la muestra de indicadores para graficar la distribución de nueva edificación y criticidad en la mancha urbana.

In [ ]:
conteo_total = resultado['tabla'].filter(ee.Filter.neq('tipologia', 'Sin edificación')).size().getInfo()
print(f"Total de celdas de 250m con edificación analizadas: {conteo_total}")

# Extraer una muestra para graficar
muestra = resultado['tabla'].filter(ee.Filter.neq('tipologia', 'Sin edificación'))\
                            .limit(1000)\
                            .reduceColumns(ee.Reducer.toList(3), ['p_nuevo', 'p_critico', 'dens_edif_ha'])\
                            .get('list').getInfo()

if muestra:
    import pandas as pd
    df = pd.DataFrame(muestra, columns=['Proporción Nueva Edificación', 'Proporción en Pendiente Crítica', 'Densidad (edif/ha)'])
    
    fig, axs = plt.subplots(1, 2, figsize=(14, 4))
    df['Proporción Nueva Edificación'].hist(bins=20, ax=axs[0], color='#d73027', edgecolor='black')
    axs[0].set_title('Distribución de Nueva Edificación (Post-2016)')
    axs[0].set_xlabel('Proporción')
    
    df['Proporción en Pendiente Crítica'].hist(bins=20, ax=axs[1], color='#6a51a3', edgecolor='black')
    axs[1].set_title('Exposición en Ladera Crítica (>= 30%)')
    axs[1].set_xlabel('Proporción')
    
    plt.tight_layout()
    plt.show()

## 11. Exportación de Resultados
Guarda la capa de indicadores en GeoJSON o GeoTIFF para incorporarla al **Atlas Municipal del MDC** y al **Atlas Urbano de la UNAH**.

In [ ]:
# Exportar a Google Drive (opcional)
task = ee.batch.Export.table.toDrive(
    collection=resultado['tabla'],
    description='MDC_indicadores_brecha_urbana',
    folder='Copernicus_BrechaCatastral',
    fileFormat='GeoJSON'
)
# Descomenta la siguiente línea para iniciar la tarea en tu Google Drive:
# task.start()
print("✔ Configuración de exportación a Google Drive lista (descomenta task.start() para lanzar).")